# BSI Examples

Three small Python examples:

1. MAB Gaussian BSI
2. sub-Gaussian/Beta MAB BSI
3. contextual linear Gaussian BSI

Each example calls the package functions directly. BSI selects the inner Monte Carlo size automatically and uses `Mmax=10000` as the compute budget.

In [ ]:
from pathlib import Path

from run import build_parser as build_mab_parser, run_config
from contextual.run_contextual import build_parser as build_contextual_parser, run_contextual_config

RESULTS = Path("example_results")
RESULTS.mkdir(exist_ok=True)


def mab_args(argv):
    args = build_mab_parser().parse_args(argv)
    if args.var_estimation_beta is None:
        args.var_estimation_beta = "hoeffding"
    return args


## 1. MAB Gaussian BSI

In [ ]:
args = mab_args([
    "--env", "normal",
    "--mus", "0.1", "0.2", "1.0",
    "--sigmas", "1.0", "1.0", "1.0",
    "--pi0", "epsilon_greedy",
    "--pi1", "ts_normal",
    "--T", "50",
    "--T_offline", "100",
    "--offline_reps", "20",
    "--Mmax", "10000",
    "--alphas", "0.10",
    "--no-save_logged_data",
    "--save_dir", str(RESULTS / "mab_gaussian"),
    "--tag", "quick_mab_gaussian",
])
run_config(args)


## 2. Sub-Gaussian/Beta MAB BSI

In [ ]:
args = mab_args([
    "--env", "beta",
    "--beta_alphas", "0.35", "0.5", "0.5",
    "--beta_betas", "0.65", "0.5", "0.5",
    "--var_estimation_beta", "hoeffding",
    "--pi0", "ts_normal",
    "--pi1", "ts_normal",
    "--T", "50",
    "--T_offline", "100",
    "--offline_reps", "20",
    "--Mmax", "10000",
    "--alphas", "0.10",
    "--no-save_logged_data",
    "--save_dir", str(RESULTS / "mab_beta_subgaussian"),
    "--tag", "quick_mab_beta",
])
run_config(args)


## 3. Contextual Linear Gaussian BSI

In [ ]:
args = build_contextual_parser().parse_args([
    "--env", "linear_gaussian",
    "--pi0", "uniform",
    "--pi1", "contextual_ts",
    "--T", "50",
    "--T_offline", "100",
    "--offline_reps", "1",
    "--lambda_star", "0.15", "1.00", "-0.75", "0.15", "1.00", "-0.75", "0.05", "-0.90", "0.95",
    "--Mmax", "10000",
    "--alpha", "0.10",
    "--save_path", str(RESULTS / "contextual_linear_gaussian.json"),
])
run_contextual_config(args)


## 4. Cluster reproduction for paper contextual experiments

The paper contextual synthetic experiments were submitted on CX3 with the helper script
`cluster/submit_contextual_natural_T500_all.sh`.

Setting used for the paper runs:

- environments: `linear_gaussian`, `logistic_bernoulli`
- policy pairs: `eps_eps` and `uni_ts`
- `T = 500`
- `T_off = 300,500,1000,2000,5000`
- outer Monte Carlo repetitions: `1000`
- truth Monte Carlo repetitions: `50000`
- natural contextual parameters: `param_scenario=natural`
- contexts: `X_t=(1,Z_{t,1},Z_{t,2})`, with `Z_t ~ N(0, I_2)`
- SVI inner Monte Carlo is selected automatically with `M_MAX=50000`, `M_TAU=0.05`, and `M_REL_EPS=0.1`
- Thompson-sampling action probabilities use `TS_PROB_MC=1000`

First upload the repository to the cluster from your local machine:

```bash
rsync -av \
  --exclude '.git/' \
  --exclude '__pycache__/' \
  --exclude '.venv/' \
  --exclude 'cluster_results*/' \
  /Users/chih-yuchang/Desktop/bsi/BanditSimulationForInference/ \
  cc1225@login.cx3.hpc.imperial.ac.uk:~/contextBSI/
```

Then on the cluster:

```bash
cd ~/contextBSI
source ~/.bashrc
micromamba activate healthkit
chmod +x cluster/submit_contextual_natural_T500_all.sh
```


Submit the `eps-eps` jobs:

```bash
PAIRS=eps_eps \
SUBMIT_TRUTH=true \
SUBMIT_BASELINES=true \
SUBMIT_BSI=true \
SELECT_INNER_REPS=true \
M_MAX=50000 \
M_TAU=0.05 \
M_REL_EPS=0.1 \
TS_PROB_MC=1000 \
BSI_MEM=20gb \
BSI_WALLTIME=03:00:00 \
RUN_PREFIX=natural_T500_selectM_cap50000_tau005_tsprob1000 \
./cluster/submit_contextual_natural_T500_all.sh
```

Submit the `uniform-TS` jobs:

```bash
PAIRS=uni_ts \
SUBMIT_TRUTH=true \
SUBMIT_BASELINES=true \
SUBMIT_BSI=true \
SELECT_INNER_REPS=true \
M_MAX=50000 \
M_TAU=0.05 \
M_REL_EPS=0.1 \
TS_PROB_MC=1000 \
BSI_MEM=20gb \
BSI_WALLTIME=03:00:00 \
RUN_PREFIX=natural_T500_selectM_cap50000_tau005_tsprob1000 \
./cluster/submit_contextual_natural_T500_all.sh
```

To run only the linear Gaussian `uniform-TS` setting, add `ENVS=linear_gaussian`:

```bash
PAIRS=uni_ts \
ENVS=linear_gaussian \
SUBMIT_TRUTH=true \
SUBMIT_BASELINES=true \
SUBMIT_BSI=true \
SELECT_INNER_REPS=true \
M_MAX=50000 \
M_TAU=0.05 \
M_REL_EPS=0.1 \
TS_PROB_MC=1000 \
BSI_MEM=20gb \
BSI_WALLTIME=03:00:00 \
RUN_PREFIX=natural_T500_selectM_cap50000_tau005_tsprob1000 \
./cluster/submit_contextual_natural_T500_all.sh
```
